# Phase 1: Raw Exploratory Data Analysis

**Purpose**: Validate downloaded WESAD dataset integrity before merging

**What to check**:
- All expected subjects are present (according to the official dataset readme)
- Pickle files are not corrupted and can therefore be loaded
- Sampling rates are consistent
- All sensors are available per subject
- Data quality issues

In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle

# Use metadata extractor
import sys

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sys.path.append("../")
from src.data.wesad_metadata_extractor import (
    determine_chest_sampling_rate,
    determine_wrist_sampling_rates,
    extract_wesad_metadata,
)

# Styling
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Imports successful!")

In [ ]:
# Dataset configuration
WESAD_PATH = Path("../data/raw/wesad/WESAD")
EXPECTED_SUBJECTS = [
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    13,
    14,
    15,
    16,
    17,
]  # S12 is missing as per official readme
LABEL_FREQ = 700  # Hz - from WESAD documentation

# Expected sensors
EXPECTED_CHEST_SENSORS = ["ACC", "ECG", "EMG", "EDA", "Temp", "Resp"]
EXPECTED_WRIST_SENSORS = ["ACC", "BVP", "EDA", "TEMP"]

print(f"WESAD Path: {WESAD_PATH.absolute()}")
print(f"Expected subjects: {len(EXPECTED_SUBJECTS)}")

## 1. Verify all expected subject folders exist

In [ ]:
print("DATASET SUBJECTS CHECK:\n")

# Check if WESAD directory exists
if not WESAD_PATH.exists():
    print(f"ERROR: WESAD directory not found at {WESAD_PATH}")
    print("Please download the dataset first")
else:
    print(f"WESAD directory found: {WESAD_PATH}\n")

    # Find all subject directories
    subject_dirs = sorted(
        [d for d in WESAD_PATH.iterdir() if d.is_dir() and d.name.startswith("S")]
    )
    found_subjects = [int(d.name[1:]) for d in subject_dirs]

    print(f"Found {len(subject_dirs)} subject directories:")
    print(f"  {[d.name for d in subject_dirs]}\n")

    # Check for missing subjects
    missing_subjects = set(EXPECTED_SUBJECTS) - set(found_subjects)
    extra_subjects = set(found_subjects) - set(EXPECTED_SUBJECTS)

    if missing_subjects:
        print(f" Missing subjects: {sorted(missing_subjects)}")
    else:
        print(" All expected subjects present")

    if extra_subjects:
        print(f" Extra subjects found: {sorted(extra_subjects)}")

    # Check for pickle files
    print("\nChecking pickle files:")
    valid_subjects = []
    missing_files = []

    for sid in found_subjects:
        pkl_path = WESAD_PATH / f"S{sid}" / f"S{sid}.pkl"
        if pkl_path.exists():
            valid_subjects.append(sid)
            print(f" S{sid}.pkl")
        else:
            missing_files.append(sid)
            print(f" S{sid}.pkl MISSING")

    print(
        f"\n✓ Valid subjects with pickle files: {len(valid_subjects)}/{len(found_subjects)}"
    )

    # Save for later use
    VALID_SUBJECTS = valid_subjects

## 2. Check for possible corruption in pickle files:

In [ ]:
print("PICKLE FILE INTEGRITY CHECK:\n")

corrupted_files = []
loadable_subjects = []

for sid in VALID_SUBJECTS:
    pkl_path = WESAD_PATH / f"S{sid}" / f"S{sid}.pkl"
    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f, encoding="latin1")

        # Basic structure check
        if "signal" in data and "label" in data:
            print(f"S{sid}: Loadable and has expected structure")
            loadable_subjects.append(sid)
        else:
            print(f" S{sid}: Missing 'signal' or 'label' key")
            corrupted_files.append(sid)

    except Exception as e:
        print(f" S{sid}: CORRUPTED - {str(e)[:50]}")
        corrupted_files.append(sid)

print("\n")
if corrupted_files:
    print(f" CORRUPTED FILES: {corrupted_files}")
    print("Dataset might need to be redownloaded.")
else:
    print(f" ALL FILES LOADABLE: {len(loadable_subjects)}/{len(VALID_SUBJECTS)}")

# Update valid subjects list
VALID_SUBJECTS = loadable_subjects

## 3. Use metadata extractor to get sampling rates and sensor info

In [ ]:
print("METADATA EXTRACTION")

# Store metadata for all subjects
metadata_db = {}

for sid in VALID_SUBJECTS:
    pkl_path = WESAD_PATH / f"S{sid}" / f"S{sid}.pkl"

    try:
        # Extract metadata
        metadata = extract_wesad_metadata(str(pkl_path))
        chest_rate = determine_chest_sampling_rate(str(pkl_path))
        wrist_rates = determine_wrist_sampling_rates(str(pkl_path), chest_rate)

        metadata_db[sid] = {
            "metadata": metadata,
            "chest_rate": chest_rate,
            "wrist_rates": wrist_rates,
            "label_length": metadata["label_length"],
        }

        print(f"S{sid}: Extracted metadata with sucess")

    except Exception as e:
        print(f"S{sid}: Failed - {str(e)[:50]}")

print(f"\nExtracted metadata for {len(metadata_db)} subjects")

## 4. Check if all subjects have the same sampling rates

In [ ]:
print("SAMPLING RATE VERIFICATION")

# Collect sampling rates
chest_rates = [
    metadata_db[sid]["chest_rate"] for sid in metadata_db
]  # expectetion: [700, 700, 700,..]
wrist_rates_by_sensor = defaultdict(list)

for sid in metadata_db:
    for sensor, rate in metadata_db[sid]["wrist_rates"].items():
        wrist_rates_by_sensor[sensor].append(rate)

# Check chest rate consistency
print("\nChest Sampling Rates:")
unique_chest_values = set(chest_rates)
if len(unique_chest_values) == 1:
    print(f"  This is consistent: {list(unique_chest_values)[0]} Hz")
else:
    print(f"     INCONSISTENT!: {unique_chest_values}")
    print(f"     Per subject: {dict(zip(metadata_db.keys(), chest_rates))}")

# Check wrist rate consistency
print("\nWrist Sampling Rates:")
for sensor, rates in wrist_rates_by_sensor.items():
    unique_rates = set(rates)
    if len(unique_rates) == 1:
        print(f"  {sensor}: Consistent at {list(unique_rates)[0]} Hz")
    else:
        print(f"     {sensor}: INCONSISTENT - {unique_rates}")
        # Show which subjects have different rates
        for sid, rate in zip(metadata_db.keys(), rates):
            if rate != min(rates):  # Show outliers
                print(f"       S{sid}: {rate} Hz")

## 5 . Find out if any and which subjects are missing sensors

In [ ]:
print("SENSOR AVAILABILITY CHECK")

# Track sensor availability
chest_sensor_availability = defaultdict(list)
wrist_sensor_availability = defaultdict(list)

for sid in metadata_db:
    # Check chest sensors
    chest_sensors = list(metadata_db[sid]["metadata"]["chest_signals"].keys())
    for sensor in EXPECTED_CHEST_SENSORS:
        chest_sensor_availability[sensor].append(sensor in chest_sensors)

    # Check wrist sensors
    wrist_sensors = list(metadata_db[sid]["metadata"]["wrist_signals"].keys())
    for sensor in EXPECTED_WRIST_SENSORS:
        wrist_sensor_availability[sensor].append(sensor in wrist_sensors)

# Print chest sensor availability
print("\nChest Sensors:")
for sensor in EXPECTED_CHEST_SENSORS:
    available = chest_sensor_availability[sensor]
    count = sum(available)
    total = len(available)

    if count == total:
        print(f"  {sensor}: {count}/{total} subjects")
    else:
        print(f"  {sensor}: {count}/{total} subjects")

        missing = []
        subject_ids = list(metadata_db.keys())

        for i in range(len(subject_ids)):
            subject_id = subject_ids[i]
            is_available = available[i]

            if not is_available:
                missing.append(subject_id)
                print(f"     Missing in: {missing}")

# Print wrist sensor availability
print("\nWrist Sensors:")
for sensor in EXPECTED_WRIST_SENSORS:
    available = wrist_sensor_availability[sensor]
    count = sum(available)
    total = len(available)

    if count == total:
        print(f"  {sensor}: {count}/{total} subjects")
    else:
        print(f"  {sensor}: {count}/{total} subjects")

        missing = []
        subject_ids = list(metadata_db.keys())

        for i in range(len(subject_ids)):
            subject_id = subject_ids[i]
            is_available = available[i]

            if not is_available:
                missing.append(subject_id)
                print(f"     Missing in: {missing}")

        print(f"     Missing in: {missing}")

## 6. Sensor Availability Heatmap

In [ ]:
# Create availability matrix for visualization
chest_matrix = np.array(
    [chest_sensor_availability[sensor] for sensor in EXPECTED_CHEST_SENSORS]
).T
wrist_matrix = np.array(
    [wrist_sensor_availability[sensor] for sensor in EXPECTED_WRIST_SENSORS]
).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Chest sensors heatmap
sns.heatmap(
    chest_matrix,
    annot=True,
    fmt="d",
    cmap="RdYlGn",
    xticklabels=EXPECTED_CHEST_SENSORS,
    yticklabels=[f"S{sid}" for sid in metadata_db],
    cbar_kws={"label": "Available"},
    ax=ax1,
)
ax1.set_title("Chest Sensor Availability", fontsize=14, fontweight="bold")
ax1.set_xlabel("Sensor")
ax1.set_ylabel("Subject")

# Wrist sensors heatmap
sns.heatmap(
    wrist_matrix,
    annot=True,
    fmt="d",
    cmap="RdYlGn",
    xticklabels=EXPECTED_WRIST_SENSORS,
    yticklabels=[f"S{sid}" for sid in metadata_db],
    cbar_kws={"label": "Available"},
    ax=ax2,
)
ax2.set_title("Wrist Sensor Availability", fontsize=14, fontweight="bold")
ax2.set_xlabel("Sensor")
ax2.set_ylabel("Subject")

plt.tight_layout()
plt.savefig(
    "../results/figures/01_sensor_availability.png", dpi=300, bbox_inches="tight"
)
plt.show()

print("✓ Heatmap saved to: results/figures/01_sensor_availability.png")

## 7. Get durations and label length per subject

In [ ]:
print("DATA LENGTH AND DURATION")

durations = []
label_lengths = []

for sid in metadata_db:
    label_length = metadata_db[sid]["label_length"]
    chest_rate = metadata_db[sid]["chest_rate"]
    duration_seconds = label_length / chest_rate
    duration_minutes = duration_seconds / 60

    durations.append(duration_minutes)
    label_lengths.append(label_length)

    print(f"S{sid}: {label_length:,} samples = {duration_minutes:.1f} minutes")

# Statistics
print("\n")
print("Duration Statistics:")
print(f"  Mean: {np.mean(durations):.1f} minutes")
print(f"  Std:  {np.std(durations):.1f} minutes")
print(
    f"  Min:  {np.min(durations):.1f} minutes (S{list(metadata_db.keys())[np.argmin(durations)]})"
)
print(
    f"  Max:  {np.max(durations):.1f} minutes (S{list(metadata_db.keys())[np.argmax(durations)]})"
)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
ax1.bar(range(len(durations)), durations, color="steelblue", alpha=0.7)
ax1.set_xlabel("Subject Index")
ax1.set_ylabel("Duration (minutes)")
ax1.set_title("Recording Duration per Subject")
ax1.axhline(
    np.mean(durations),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(durations):.1f} min",
)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histogram
ax2.hist(durations, bins=10, color="coral", alpha=0.7, edgecolor="black")
ax2.set_xlabel("Duration (minutes)")
ax2.set_ylabel("Frequency")
ax2.set_title("Distribution of Recording Durations")
ax2.axvline(
    np.mean(durations),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(durations):.1f} min",
)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "../results/figures/01_recording_durations.png", dpi=300, bbox_inches="tight"
)
plt.show()

## 8. Load a sample subject and quick check for data quality issues

In [ ]:
print("SAMPLE DATA QUALITY CHECK (S4)")

# Load subject 4 as example
sample_path = WESAD_PATH / "S4" / "S4.pkl"

with open(sample_path, "rb") as f:
    sample_data = pickle.load(f, encoding="latin1")

# Check labels
print("\nLabel Distribution:")
labels, counts = np.unique(sample_data["label"], return_counts=True)
label_names = {
    0: "Not defined / Transient",
    1: "Baseline",
    2: "Stress",
    3: "Amusement",
    4: "Meditation",
    5: "Should be ignored (per README)",
    6: "Should be ignored (per README)",
    7: "Should be ignored (per README)",
}

for label, count in zip(labels, counts):
    pct = (count / len(sample_data["label"])) * 100
    print(
        f"  {int(label)}: {label_names.get(int(label), 'Unknown'):20s} - {count:8,} samples ({pct:5.1f}%)"
    )

# Check for obvious issues in chest temperature
print("\nChest Temperature Check:")
chest_temp = sample_data["signal"]["chest"]["Temp"]
print(f"  Min: {chest_temp.min():.2f}°C")
print(f"  Max: {chest_temp.max():.2f}°C")
print(f"  Mean: {chest_temp.mean():.2f}°C")

invalid_temps = (chest_temp <= 0).sum()
if invalid_temps > 0:
    print(f"   Found {invalid_temps:,} invalid temperatures (≤0)")
else:
    print("   No invalid temperatures detected")

# Check wrist temperature
print("\nWrist Temperature Check:")
wrist_temp = sample_data["signal"]["wrist"]["TEMP"]
print(f"  Min: {wrist_temp.min():.2f}°C")
print(f"  Max: {wrist_temp.max():.2f}°C")
print(f"  Mean: {wrist_temp.mean():.2f}°C")

invalid_temps = (wrist_temp <= 0).sum()
if invalid_temps > 0:
    print(f"   Found {invalid_temps:,} invalid temperatures (≤0)")
else:
    print("   No invalid temperatures detected")

## 9. Summary

In [ ]:
print("STAGE 1 EDA SUMMARY REPORT")

print("\n DATASET OVERVIEW:")
print(f"  • Expected subjects: {len(EXPECTED_SUBJECTS)}")
print(f"  • Found subjects: {len(VALID_SUBJECTS)}")
print(f"  • Loadable subjects: {len(metadata_db)}")

print("\n DATA QUALITY:")
all_chest_complete = all(
    all(chest_sensor_availability[s]) for s in EXPECTED_CHEST_SENSORS
)
all_wrist_complete = all(
    all(wrist_sensor_availability[s]) for s in EXPECTED_WRIST_SENSORS
)

if all_chest_complete:
    print("    All chest sensors present in all subjects")
else:
    print("    Some chest sensors missing in some subjects")

if all_wrist_complete:
    print("    All wrist sensors present in all subjects")
else:
    print("    Some wrist sensors missing in some subjects")

# Check sampling rate consistency
chest_consistent = len(set(chest_rates)) == 1
wrist_consistent = all(
    len(set(wrist_rates_by_sensor[s])) == 1 for s in wrist_rates_by_sensor
)

if chest_consistent:
    print(f"    Chest sampling rates consistent: {chest_rates[0]} Hz")
else:
    print("    Chest sampling rates inconsistent")

if wrist_consistent:
    print("    Wrist sampling rates consistent")
else:
    print("    Wrist sampling rates inconsistent")

print("\n RECORDING INFO:")
print(f"  • Mean duration: {np.mean(durations):.1f} minutes")
print(
    f"  • Total recording time: {sum(durations):.1f} minutes ({sum(durations)/60:.1f} hours)"
)

print("\n")
if all_chest_complete and all_wrist_complete and chest_consistent and wrist_consistent:
    print("✅DATASET READY FOR MERGING")
else:
    print("ISSUES DETECTED! Review before proceeding")